# 01 — Data Ingestion & Source Merging
**Loads all three article sources, standardises schemas, and writes a clean unified CSV to `data/processed/`.**

| Source | File | Schema |
|--------|------|--------|
| Economic Times | `et_banking_news_slim.csv` | ticker, date, source, headline, text |
| Moneycontrol | `mc_articles.csv` | ticker, date, source, title |
| Guardian API | `guardian_banking_news.csv` | ticker, date, source, headline, text |

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

ROOT = Path('..') 
RAW  = ROOT / 'data' / 'raw'
PROC = ROOT / 'data' / 'processed'
PROC.mkdir(parents=True, exist_ok=True)

print('Directories ready.')

Directories ready.


## 1. Load Economic Times (ET)

In [2]:
et_df = pd.read_csv(RAW / 'et_banking_news_slim.csv')
print(f'ET shape: {et_df.shape}')
et_df.head(3)

ET shape: (630, 5)


,ticker,date,source,headline,text
0,HDFCBANK,2024-09-18 15:30:00,Economic Times,"US Fed rate policy decision today: Check time,...","US Fed rate policy decision today: Check time,..."
1,ICICIBANK,2024-09-18 15:30:00,Economic Times,"US Fed rate policy decision today: Check time,...","US Fed rate policy decision today: Check time,..."
2,SBIN,2024-09-18 15:30:00,Economic Times,"US Fed rate policy decision today: Check time,...","US Fed rate policy decision today: Check time,..."


In [3]:
# Standardise ET → unified schema
et_clean = pd.DataFrame({
    'ticker' : et_df['ticker'].str.upper().str.strip(),
    'date'   : pd.to_datetime(et_df['date']).dt.normalize(),
    'source' : et_df['source'].fillna('Economic Times'),
    'headline': et_df['headline'].fillna(''),
    'text'   : et_df['text'].fillna('')
})
print(f'ET cleaned: {et_clean.shape}')
et_clean.dtypes

ET cleaned: (630, 5)


ticker                 str
date        datetime64[us]
source                 str
headline               str
text                   str
dtype: object

## 2. Load Moneycontrol (MC)

In [4]:
mc_df = pd.read_csv(RAW / 'mc_articles.csv')
print(f'MC shape: {mc_df.shape}')
mc_df.head(3)

MC shape: (2923, 4)


,ticker,date,source,title
0,axis,2026-03-19,moneycontrol,Axis Bank to invest $162 million in consumer l...
1,axis,2026-03-18,moneycontrol,"Axis Bank to infuse Rs 1,500 crore in consumer..."
2,axis,2026-03-16,moneycontrol,NCDRC orders Axis Bank to pay Rs 3.19 crore ov...


In [5]:
# MC uses 'title' instead of 'headline', and lowercase tickers
# Map MC tickers → NSE symbols
MC_TICKER_MAP = {
    'hdfc'    : 'HDFCBANK',
    'icici'   : 'ICICIBANK',
    'sbi'     : 'SBIN',
    'axis'    : 'AXISBANK',
    'kotak'   : 'KOTAKBANK',
    'indusind': 'INDUSINDBK',
}

mc_clean = pd.DataFrame({
    'ticker' : mc_df['ticker'].str.lower().str.strip().map(MC_TICKER_MAP).fillna(mc_df['ticker'].str.upper()),
    'date'   : pd.to_datetime(mc_df['date']).dt.normalize(),
    'source' : mc_df['source'].fillna('Moneycontrol'),
    'headline': mc_df['title'].fillna(''),
    'text'   : ''   # MC does not supply full text
})
print(f'MC cleaned: {mc_clean.shape}')
print('Tickers:', mc_clean.ticker.unique())

MC cleaned: (2923, 5)
Tickers: <StringArray>
['AXISBANK', 'HDFCBANK', 'ICICIBANK', 'INDUSINDBK', 'KOTAKBANK', 'SBIN']
Length: 6, dtype: str


## 3. Load Guardian API data

In [6]:
guardian_path = RAW / 'guardian_banking_news_slim.csv'

if guardian_path.exists():
    g_df = pd.read_csv(guardian_path)
    print(f'Guardian shape: {g_df.shape}')
    print('Columns:', g_df.columns.tolist())
    g_df.head(3)
else:
    print('⚠️  guardian_banking_news.csv not found — place your Guardian-scraped file in data/raw/')
    print('Expected schema: ticker, date, source, headline, text')
    # Create empty placeholder so downstream cells still work
    g_df = pd.DataFrame(columns=['ticker','date','source','headline','text'])

Guardian shape: (10113, 5)
Columns: ['ticker', 'date', 'source', 'headline', 'text']


In [7]:
if not g_df.empty:
    guardian_clean = pd.DataFrame({
        'ticker' : g_df['ticker'].str.upper().str.strip(),
        'date'   : pd.to_datetime(g_df['date']).dt.normalize(),
        'source' : g_df.get('source', pd.Series('Guardian')).fillna('Guardian'),
        'headline': g_df['headline'].fillna(''),
        'text'   : g_df.get('text', pd.Series('')).fillna('')
    })
else:
    guardian_clean = pd.DataFrame(columns=['ticker','date','source','headline','text'])

print(f'Guardian cleaned: {guardian_clean.shape}')

Guardian cleaned: (10113, 5)


## 4. Merge & deduplicate

In [8]:
all_frames = [et_clean, mc_clean]
if not guardian_clean.empty:
    # Remove timezone from Guardian data before concatenating
    guardian_clean['date'] = pd.to_datetime(guardian_clean['date']).dt.tz_localize(None)
    all_frames.append(guardian_clean)

merged = pd.concat(all_frames, ignore_index=True)
print(f'Merged (before dedup): {merged.shape}')

# Ensure all dates are timezone-naive
merged['date'] = pd.to_datetime(merged['date'], utc=False)

# Drop exact duplicate (ticker + date + headline)
merged.drop_duplicates(subset=['ticker','date','headline'], inplace=True)
merged.sort_values(['date','ticker'], inplace=True)
merged.reset_index(drop=True, inplace=True)

print(f'After dedup: {merged.shape}')
merged.head()

Merged (before dedup): (13666, 5)
After dedup: (13649, 5)


,ticker,date,source,headline,text
0,SBIN,2022-02-07,moneycontrol,What should investors do with SBI after Q3 res...,
1,SBIN,2022-02-08,moneycontrol,SBI Consolidated December 2021 Net Interest In...,
2,SBIN,2022-02-08,moneycontrol,SBI Standalone December 2021 Net Interest Inco...,
3,SBIN,2022-02-10,moneycontrol,RBI Monetary Policy: MPC keeps rates unchanged...,
4,SBIN,2022-02-12,moneycontrol,"CBI books ABG Shipyard, former top brass in bi...",


In [9]:
print('Date range:', merged.date.min(), '->', merged.date.max())
print('\nArticles per source:')
print(merged.source.value_counts())
print('\nArticles per ticker:')
print(merged.ticker.value_counts())

Date range: 2022-02-07 00:00:00 -> 2026-03-24 00:00:00

Articles per source:
source
The Guardian      10113
moneycontrol       2918
Economic Times      618
Name: count, dtype: int64

Articles per ticker:
ticker
HDFCBANK      2917
ICICIBANK     2368
INDUSINDBK    2272
KOTAKBANK     2226
SBIN          1939
AXISBANK      1927
Name: count, dtype: int64


## 5. Save unified dataset

In [10]:
out_path = PROC / 'all_banking_news.csv'
merged.to_csv(out_path, index=False)
print(f'✅ Saved {len(merged):,} articles → {out_path}')

✅ Saved 13,649 articles → ..\data\processed\all_banking_news.csv
